# Example 6: Training a Virtual Room for System Equalization

This notebook demonstrates how to train a virtual room (a set of FIR filters) to equalize a Reverberation Enhancement System (RES). The goal is to shape the eigenvalues of the open‑loop transfer matrix so that they follow a desired target curve, thereby controlling the system’s stability and colouration.

**Training pipeline** (based on De Bortoli et al., DAFx 2024):
1. Create a physical room (using white‑noise RIRs) and a virtual room (random FIRs).
2. Build the RES and extract its open‑loop model.
3. Generate a dataset: unit impulses as inputs, and a target equalisation curve derived from the initial eigenvalues.
4. Train the virtual room’s FIR coefficients by minimising the mean‑squared error between the current open‑loop eigenvalues and the target.
5. Compare the eigenvalue distribution and impulse responses before and after optimisation.

The hyperparameters (dataset size, learning rate, etc.) are defined in the first code cell – adjust them as needed.

## 1. Imports and Setup

In [ ]:
import sys
import os
import time
# Add parent directory to path (to find PyRES)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
import matplotlib.pyplot as plt

from flamo import system, dsp
from flamo.optimize.dataset import Dataset, load_dataset
from flamo.optimize.trainer import Trainer

from PyRES.res import RES
from PyRES.physical_room import PhRoom_wgn
from PyRES.virtual_room import random_FIRs
from PyRES.loss_functions import MSE_evs_mod
from PyRES.functional import system_equalization_curve
from PyRES.plots import plot_evs_compare, plot_spectrograms_compare

# Set random seed for reproducibility
torch.manual_seed(141122)

## 2. Hyperparameters

These variables replace the command‑line arguments. Change them to experiment with different settings.

In [ ]:
# ----------------------- Dataset -----------------------
num = 2**5                # dataset size (number of training examples)
device = 'cpu'             # 'cpu' or 'cuda' if available
split = 0.8                # train/validation split ratio

# ---------------------- Training -----------------------
max_epochs = 10            # maximum number of epochs
patience_delta = 1e-4      # minimum improvement to reset patience
lr = 1e-3                  # learning rate

# ---------------------- Output -------------------------
train_dir = None           # set to a path to save training logs; if None, a timestamped folder is created

## 3. Create the Physical and Virtual Rooms

In [ ]:
# Time‑frequency parameters
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # anti‑aliasing decay (dB)

# Physical room (stochastic)
room_dims = (12.1, 8.5, 3.2)   # length, width, height (m)
room_RT = 0.7                   # reverberation time (s)
n_M = 4                         # number of microphones
n_L = 8                         # number of loudspeakers

physical_room = PhRoom_wgn(
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    room_dims=room_dims,
    room_RT=room_RT,
    n_M=n_M,
    n_L=n_L
)

# Virtual room: random FIR filters (trainable)
fir_order = 2**8                # filter length
virtual_room = random_FIRs(
    n_M=n_M,
    n_L=n_L,
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    FIR_order=fir_order,
    requires_grad=True
)

# Build the RES
res = RES(physical_room=physical_room, virtual_room=virtual_room)

## 4. Define the Model (Open Loop)

In [ ]:
model = system.Shell(
    core=res.open_loop(),
    input_layer=system.Series(
        dsp.FFT(nfft=nfft),
        dsp.Transform(lambda x: x.diag_embed())
    )
)

## 5. Initial Performance (Before Training)

Compute the open‑loop eigenvalues and the system impulse responses at initialization for later comparison.

In [ ]:
evs_init = res.open_loop_eigenvalues()
_, _, ir_init = res.system_simulation()

## 6. Create the Dataset

Input: a single impulse per channel (at sample 0).  
Target: an equalisation curve derived from the initial eigenvalues, cut off at 8 kHz.

In [ ]:
dataset_input = torch.zeros(1, samplerate, n_M)
dataset_input[:, 0, :] = 1.0

target_curve = system_equalization_curve(evs=evs_init, fs=samplerate, nfft=nfft, f_c=8000)
dataset_target = target_curve.view(1, -1, 1).expand(1, -1, n_M)   # shape: (1, freq_bins, n_M)

dataset = Dataset(
    input=dataset_input,
    target=dataset_target,
    expand=num,
    device=device
)

train_loader, valid_loader = load_dataset(dataset, batch_size=1, split=split, shuffle=False)

## 7. Set Up the Trainer and Loss Function

In [ ]:
# Prepare output directory
if train_dir is None:
    train_dir = os.path.join('training_output', time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(train_dir, exist_ok=True)

# Save hyperparameters to a text file (optional)
with open(os.path.join(train_dir, 'args.txt'), 'w') as f:
    f.write('\n'.join([f"{k},{v}" for k, v in locals().items() if k in ['num','device','split','max_epochs','patience_delta','lr']]))

trainer = Trainer(
    net=model,
    max_epochs=max_epochs,
    lr=lr,
    patience_delta=patience_delta,
    train_dir=train_dir,
    device=device
)

# Loss function: MSE on eigenvalues with frequency weighting
criterion = MSE_evs_mod(
    iter_num=num,
    freq_points=nfft // 2 + 1,
    samplerate=samplerate,
    lowest_f=20,
    highest_f=15000
)
trainer.register_criterion(criterion, 1.0)

## 8. Run Training

In [ ]:
trainer.train(train_loader, valid_loader)

## 9. Evaluate After Training

In [ ]:
evs_opt = res.open_loop_eigenvalues()
_, _, ir_opt = res.system_simulation()

# Compare eigenvalues before and after
plot_evs_compare(evs_init, evs_opt, samplerate, nfft, 20, 8000)
plt.show()

# Compare impulse responses (first channel)
plot_spectrograms_compare(
    ir_init[:, 0],
    ir_opt[:, 0],
    fs=samplerate,
    nfft=2**11,
    noverlap=2**10
)
plt.show()

## 10. (Optional) Save the Trained Virtual Room

Uncomment the following line to save the virtual room parameters. They can later be reloaded into an identical virtual room instance to skip retraining.

In [ ]:
# res.save_state_to(directory='./model_states/')

## 11. Conclusion

You have successfully trained a virtual room to shape the open‑loop eigenvalues toward a desired equalisation target. This is a proof‑of‑concept; increasing the dataset size (`num`) and training for more epochs will improve the result. For further details, refer to the paper:

> De Bortoli, G., Dal Santo, G., Prawda, K., Lokki, T., Välimäki, V., and Schlecht, S. J.  
> "Differentiable Active Acoustics: Optimizing Stability via Gradient Descent"  
> *Proceedings of the International Conference on Digital Audio Effects*, pp. 254‑261, 2024.